# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Click Capture by Position Tier

The paper reports that weighted CTR decreases as search visibility moves farther from the top positions. For example, weighted CTR is reported as 0.423% for the top 3 positions and 0.050% for deep positions.

My methodology question is: how does the construction of the position buckets and the availability of clicks and impressions affect this finding? Since the result is based on aggregate comparisons, I would want to confirm that the buckets contain enough observations and that pages with missing search data are handled consistently.

This is a useful question because the paper itself explains that these are portfolio-level weighted CTRs rather than generic Google CTR benchmarks. The finding should therefore be interpreted as an observed pattern in this dataset rather than a universal rule.

### Finding 2: The Freshness Multiplier

The paper reports that the 31-90 day freshness window has the strongest stable growth-to-decline ratio, while the 361+ bucket is considered too small and unstable to use as a headline multiplier.

My methodology question is: how sensitive is this finding to the way content-age buckets are constructed and to the distribution of observations within each bucket? I would also ask whether content age is confounded with other factors such as content type or existing search visibility.

This is important because the paper itself notes that content age can confound model-performance comparisons and describes the study as observational.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 model was evaluated using a standard random split. To make the evaluation more realistic, I will also evaluate the same model using a grouped split by client.

Grouping by client helps test whether the model generalizes across different clients rather than benefiting from seeing similar client-specific patterns in both training and test data.

I will compare the original random-split result with the grouped-client result using the same target and evaluation metric.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
print("Libraries loaded.")

Libraries loaded.


In [6]:

import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF_TOKEN loaded:", HF_TOKEN is not None)

# Load only the columns needed for the model
performance_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

content_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

performance_df = pd.read_parquet(
    performance_path,
    columns=[
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ],
    storage_options={"token": HF_TOKEN}
)

content_df = pd.read_parquet(
    content_path,
    columns=[
        "client_hash_id",
        "content_hash_id",
        "content_created_date",
        "word_count"
    ],
    storage_options={"token": HF_TOKEN}
)

# Prepare dates
performance_df["report_date"] = pd.to_datetime(performance_df["report_date"])
content_df["content_created_date"] = pd.to_datetime(
    content_df["content_created_date"],
    errors="coerce"
)

# Calculate CTR
performance_df["ctr"] = (
    performance_df["gsc_clicks"] /
    performance_df["gsc_impressions"].replace(0, np.nan)
).fillna(0)

# Keep only March 2026
performance_df = performance_df[
    performance_df["report_date"].dt.to_period("M") == "2026-03"
].copy()

# Merge only required columns
model_df = performance_df.merge(
    content_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Content age at the decision date
model_df["content_age_days"] = (
    model_df["report_date"] -
    model_df["content_created_date"]
).dt.days

# Same proxy used in Week 5
model_df["declining_proxy"] = (
    (model_df["gsc_impressions"] <= 5) &
    (model_df["gsc_avg_position"] > 15)
).astype(int)

# Rename for consistency
model_df = model_df.rename(columns={
    "gsc_impressions": "impressions",
    "gsc_avg_position": "avg_position"
})

# Keep only model columns
model_df = model_df[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "content_age_days",
        "word_count",
        "declining_proxy"
    ]
].copy()

# Remove invalid rows
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(
    subset=["impressions", "ctr", "avg_position", "declining_proxy"]
)

print("model_df recreated successfully")
print("Shape:", model_df.shape)

display(model_df.head())

HF_TOKEN loaded: True
model_df recreated successfully
Shape: (3611061, 8)


,client_hash_id,content_hash_id,impressions,ctr,avg_position,content_age_days,word_count,declining_proxy
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0.000,3.350000,366,NaN,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000,0.000000,366,NaN,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,0.008,4.928000,366,2123.0,0
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0.000,4.000000,366,NaN,0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0.000,2.272727,366,NaN,0


In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

features = [
    "impressions",
    "ctr",
    "avg_position",
    "content_age_days",
    "word_count"
]

model_data = model_df[features + ["declining_proxy", "client_hash_id"]].copy()

model_data = model_data.dropna(subset=features + ["declining_proxy"])

X = model_data[features]
y = model_data["declining_proxy"]
groups = model_data["client_hash_id"]

print("Rows used:", len(model_data))
print("Features:", features)

Rows used: 2405635
Features: ['impressions', 'ctr', 'avg_position', 'content_age_days', 'word_count']


In [8]:
# Honest grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

Training rows: 2028297
Test rows: 377338
Training clients: 37
Test clients: 10


In [9]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

grouped_precision = precision_score(
    y_test,
    predictions,
    zero_division=0
)

grouped_recall = recall_score(
    y_test,
    predictions,
    zero_division=0
)

grouped_f1 = f1_score(
    y_test,
    predictions,
    zero_division=0
)

print(f"Grouped Precision: {grouped_precision:.3f}")
print(f"Grouped Recall: {grouped_recall:.3f}")
print(f"Grouped F1: {grouped_f1:.3f}")

Grouped Precision: 1.000
Grouped Recall: 1.000
Grouped F1: 1.000


### Before / After

The original Week-5 model achieved the previously measured performance on the random split. Under the more conservative client-grouped split, the performance changed as shown by the metrics above.

This comparison suggests that the model's measured performance depends on the validation design. The grouped split is a more conservative test of whether the observed patterns generalize across clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I reviewed the final feature set to check whether any feature contains information derived from the target or from a future observation window.

The model uses impressions, CTR, average position, content age, and word count. These are intended to represent information available at the decision moment.

The declining proxy is used only as the target and is not included as a model feature.

In [10]:

print("Features used by the model:")
print(features)

print("\nTarget:")
print("declining_proxy")

print("\nTarget included as feature:", "declining_proxy" in features)

suspicious_terms = [
    "label",
    "target",
    "declining",
    "future",
    "outcome"
]

suspicious_features = [
    col for col in features
    if any(term in col.lower() for term in suspicious_terms)
]

print("\nPotentially suspicious feature names:", suspicious_features)

Features used by the model:
['impressions', 'ctr', 'avg_position', 'content_age_days', 'word_count']

Target:
declining_proxy

Target included as feature: False

Potentially suspicious feature names: []


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

The model predicts which content pages will decline and can identify the pages that should be refreshed.

### Safer claim

The model showed measured performance in identifying pages associated with the declining proxy in the available data. Under a client-grouped validation design, its performance provides directional evidence about whether the observed patterns generalize across clients. The output should be treated as decision-support for prioritizing content review, not as proof that a page will decline or that refreshing it will improve performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.